## 掩码预训练

In [1]:
from transformers import AutoTokenizer,AutoModelForMaskedLM,DataCollatorForLanguageModeling,Trainer,TrainingArguments
from datasets import load_dataset

In [3]:

ds = load_dataset("pleisto/wikipedia-cn-20230720-filtered")

c:\Users\32721\anaconda3\envs\transformers\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\32721\.cache\huggingface\hub\datasets--pleisto--wikipedia-cn-20230720-filtered. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating train split: 100%|██████████| 254547/254547 [00:04<00:00, 56988.11 example

In [7]:
ds
dataset = ds["train"]
dataset

Dataset({
    features: ['completion', 'source'],
    num_rows: 254547
})

In [8]:
dataset[0]

{'completion': '昭通机场（ZPZT）是位于中国云南昭通的民用机场，始建于1935年，1960年3月开通往返航班“昆明－昭通”，原来属军民合用机场。1986年机场停止使用。1991年11月扩建，于1994年2月恢复通航。是西南地区「文明机场」，通航城市昆明。 机场占地1957亩，飞行区等级为4C，有一条跑道，长2720米，宽48米，可供波音737及以下机型起降。机坪面积6600平方米，停机位2个，航站楼面积1900平方米。位于城东6公里处，民航路与金鹰大道交叉处。\n航点\n客服电话\n昭通机场客服电话：0870-2830004',
 'source': 'wikipedia.zh2307'}

## 数据集处理

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("hfl/chinese-macbert-base")

def process_func(examles):
    return tokenizer(examles["completion"], max_length=384, truncation= True)

In [10]:
tokenized_dataset = dataset.map(process_func, batched= True, remove_columns= dataset.column_names)

Map: 100%|██████████| 254547/254547 [02:19<00:00, 1820.90 examples/s]


In [11]:
tokenizer.mask_token , tokenizer.mask_token_id

('[MASK]', 103)

## 创建模型

In [12]:
model = AutoModelForMaskedLM.from_pretrained("hfl/chinese-macbert-base")

Some weights of the model checkpoint at hfl/chinese-macbert-base were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


## 训练参数

In [13]:
args = TrainingArguments(
    output_dir= "./masked_llm",
    per_device_train_batch_size= 32,
    logging_steps= 10,
    num_train_epochs= 1
)

## 创建trainer

In [14]:
trainer = Trainer(args= args , model= model , train_dataset= tokenized_dataset , 
                  data_collator= DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm= True , mlm_probability= 0.15))

## 模型训练

In [ ]:
trainer.train()

## 模型推理

In [16]:
from transformers import pipeline

pipe = pipeline("fill-mask",model= model , tokenizer= tokenizer)

Device set to use cpu


In [17]:
pipe("华南理工大[MASK]真不错")

[{'score': 0.6573415994644165,
  'token': 6820,
  'token_str': '还',
  'sequence': '华 南 理 工 大 还 真 不 错'},
 {'score': 0.0668335035443306,
  'token': 738,
  'token_str': '也',
  'sequence': '华 南 理 工 大 也 真 不 错'},
 {'score': 0.06123834848403931,
  'token': 4638,
  'token_str': '的',
  'sequence': '华 南 理 工 大 的 真 不 错'},
 {'score': 0.028549067676067352,
  'token': 1377,
  'token_str': '可',
  'sequence': '华 南 理 工 大 可 真 不 错'},
 {'score': 0.02476358413696289,
  'token': 2110,
  'token_str': '学',
  'sequence': '华 南 理 工 大 学 真 不 错'}]